<a href="https://colab.research.google.com/github/vineetsalar88/ResearchPaper2/blob/master/March2026/29MarchConvNeXt_cleanLab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [10]:
!pip install cleanlab

In [22]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import torchvision.models as models

from torch.utils.data import DataLoader, Dataset, Subset, ConcatDataset
from sklearn.metrics import accuracy_score
import numpy as np
from cleanlab.filter import find_label_issues

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_dir = "/content/drive/MyDrive/ResearchData/March26/DatasetTrainVsTestCropped1128/train"
val_dir =   "/content/drive/MyDrive/ResearchData/March26/DatasetTrainVsTestCropped1128/val"
IMG_SIZE = 224
BATCH_SIZE = 32
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])

train_dataset = datasets.ImageFolder(train_dir, transform=transform)
val_dataset = datasets.ImageFolder(val_dir, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

NUM_CLASSES = len(train_dataset.classes)

In [12]:
def get_model(num_classes):
    model = models.convnext_tiny(pretrained=True)
    model.classifier[2] = nn.Linear(model.classifier[2].in_features, num_classes)
    return model.to(device)

In [13]:
def train_eval(model, train_loader, val_loader, epochs=5):
    optimizer = optim.AdamW(model.parameters(), lr=3e-4)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):

        # TRAIN
        model.train()
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()

        # VALIDATION
        model.eval()
        all_preds, all_labels = [], []

        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)

                outputs = model(images)
                preds = torch.argmax(outputs, 1)

                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.numpy())

        acc = accuracy_score(all_labels, all_preds)
        print(f"Epoch {epoch+1} Acc: {acc:.4f}")

    return acc

In [14]:
model = get_model(NUM_CLASSES)

print("\n🔵 Training BEFORE CleanLab")
acc_before = train_eval(model, train_loader, val_loader, epochs=5)
print("Accuracy BEFORE:", acc_before)


🔵 Training BEFORE CleanLab
Epoch 1 Acc: 0.5868
Epoch 2 Acc: 0.6257
Epoch 3 Acc: 0.6826
Epoch 4 Acc: 0.6826
Epoch 5 Acc: 0.6377
Accuracy BEFORE: 0.6377245508982036


In [15]:
model.eval()

all_probs = []
all_labels = []

with torch.no_grad():
    for images, labels in DataLoader(train_dataset, batch_size=32):
        images = images.to(device)

        probs = torch.softmax(model(images), dim=1)

        all_probs.append(probs.cpu().numpy())
        all_labels.append(labels.numpy())

pred_probs = np.concatenate(all_probs)
labels = np.concatenate(all_labels)

In [16]:
issue_indices = find_label_issues(labels=labels, pred_probs=pred_probs)

print("Noisy samples:", len(issue_indices))

Noisy samples: 778


In [17]:
corrected_labels = np.argmax(pred_probs, axis=1)

for idx in issue_indices:
    train_dataset.targets[idx] = corrected_labels[idx]

In [18]:
model_clean = get_model(NUM_CLASSES)

print("\n🟢 Training AFTER CleanLab")
acc_after_clean = train_eval(model_clean, train_loader, val_loader, epochs=5)

print("Accuracy AFTER CleanLab:", acc_after_clean)


🟢 Training AFTER CleanLab
Epoch 1 Acc: 0.5419
Epoch 2 Acc: 0.6108
Epoch 3 Acc: 0.6587
Epoch 4 Acc: 0.6138
Epoch 5 Acc: 0.6677
Accuracy AFTER CleanLab: 0.6676646706586826


In [19]:
class TripletDataset(Dataset):
    def __init__(self, dataset):
        self.dataset = dataset
        self.targets = dataset.targets

        self.class_to_indices = {}
        for i, label in enumerate(self.targets):
            self.class_to_indices.setdefault(label, []).append(i)

    def __getitem__(self, idx):
        anchor, label = self.dataset[idx]

        pos_idx = random.choice(self.class_to_indices[label])

        neg_label = random.choice(list(self.class_to_indices.keys()))
        while neg_label == label:
            neg_label = random.choice(list(self.class_to_indices.keys()))

        neg_idx = random.choice(self.class_to_indices[neg_label])

        pos, _ = self.dataset[pos_idx]
        neg, _ = self.dataset[neg_idx]

        return anchor, pos, neg

    def __len__(self):
        return len(self.dataset)

In [20]:
class EmbeddingModel(nn.Module):
    def __init__(self):
        super().__init__()
        base = models.convnext_tiny(pretrained=True)
        self.features = base.features
        self.pool = nn.AdaptiveAvgPool2d((1,1))
        self.fc = nn.Linear(768, 256)

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        return self.fc(x)

In [23]:
triplet_loader = DataLoader(TripletDataset(train_dataset), batch_size=32, shuffle=True)

embed_model = EmbeddingModel().to(device)

optimizer = optim.AdamW(embed_model.parameters(), lr=3e-4)
criterion = nn.TripletMarginLoss(margin=1.0)

for epoch in range(5):
    total_loss = 0

    for a, p, n in triplet_loader:
        a, p, n = a.to(device), p.to(device), n.to(device)

        optimizer.zero_grad()

        loss = criterion(embed_model(a), embed_model(p), embed_model(n))
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Triplet Epoch {epoch+1}: {total_loss/len(triplet_loader):.4f}")

Triplet Epoch 1: 0.6358
Triplet Epoch 2: 0.5561
Triplet Epoch 3: 0.4787
Triplet Epoch 4: 0.4174
Triplet Epoch 5: 0.3546


In [24]:
class FinalModel(nn.Module):
    def __init__(self, embed_model, num_classes):
        super().__init__()
        self.embed = embed_model
        self.fc = nn.Linear(256, num_classes)

    def forward(self, x):
        return self.fc(self.embed(x))

final_model = FinalModel(embed_model, NUM_CLASSES).to(device)

print("\n🔥 Training FINAL MODEL (Triplet + CleanLab)")
acc_final = train_eval(final_model, train_loader, val_loader, epochs=5)

print("Final Accuracy:", acc_final)


🔥 Training FINAL MODEL (Triplet + CleanLab)
Epoch 1 Acc: 0.4760
Epoch 2 Acc: 0.6557
Epoch 3 Acc: 0.5329
Epoch 4 Acc: 0.6287
Epoch 5 Acc: 0.6707
Final Accuracy: 0.6706586826347305


In [25]:
print("\n========== FINAL RESULTS ==========")
print(f"Before CleanLab      : {acc_before:.4f}")
print(f"After CleanLab       : {acc_after_clean:.4f}")
print(f"After Triplet + CleanLab : {acc_final:.4f}")


========== FINAL RESULTS ==========
Before CleanLab      : 0.6377
After CleanLab       : 0.6677
After Triplet + CleanLab : 0.6707
